# ID3 Decision Tree – Weather Forecast Dataset

formulas

EntropyCalculates H(S) = −Σ pᵢ log₂(pᵢ) to measure dataset impurity

Info GainIG(S,A) = H(S) − weighted child entropy; picks the best split feature

In [1]:
# Importing Libraries
import pandas as pd
import math
from collections import Counter
from sklearn.tree import DecisionTreeClassifier, export_text 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, classification_report 
from sklearn.preprocessing import LabelEncoder 

In [5]:
# Loading the  Dataset
df = pd.read_csv('weather_forecast_data.csv')
print('Shape:', df.shape)
df.head()

Shape: (2500, 6)


,Temperature,Humidity,Wind_Speed,Cloud_Cover,Pressure,Rain
0,23.720338,89.592641,7.335604,50.501694,1032.378759,rain
1,27.879734,46.489704,5.952484,4.990053,992.614190,no rain
2,25.069084,83.072843,1.371992,14.855784,1007.231620,no rain
3,23.622080,74.367758,7.050551,67.255282,982.632013,rain
4,20.591370,96.858822,4.643921,47.676444,980.825142,no rain


In [6]:
# Entropy & Information Gain calulation
def entropy(labels):
    total = len(labels)
    return -sum((c/total) * math.log2(c/total) for c in Counter(labels).values())

def info_gain(df, feature, target='Rain'):
    base = entropy(df[target])
    vals = df[feature].unique()
    weighted = sum((len(df[df[feature]==v]) / len(df)) * entropy(df[df[feature]==v][target]) for v in vals)
    return round(base - weighted, 4)

# Discretize features into bins for ID3
df_d = df.copy()
for col in ['Temperature','Humidity','Wind_Speed','Cloud_Cover','Pressure']:
    df_d[col] = pd.cut(df[col], bins=3, labels=['Low','Medium','High'])

features = ['Temperature','Humidity','Wind_Speed','Cloud_Cover','Pressure']
print('Root Entropy:', round(entropy(df_d['Rain']), 4))
print('\nInformation Gain per Feature:')
for f in features:
    print(f'  {f:<15} IG = {info_gain(df_d, f)}')  

Root Entropy: 0.5452

Information Gain per Feature:
  Temperature     IG = 0.0834
  Humidity        IG = 0.1162
  Wind_Speed      IG = 0.0001
  Cloud_Cover     IG = 0.0902
  Pressure        IG = 0.0002


In [7]:
# Training the ID3 Tree 
X = df[features]
y = LabelEncoder().fit_transform(df['Rain'])   # rain=1, no rain=0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)
model.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)

In [16]:
# Tree Structure
print(export_text(model, feature_names=features)) 

|--- Humidity <= 70.09
|   |--- class: 0
|--- Humidity >  70.09
|   |--- Cloud_Cover <= 50.03
|   |   |--- class: 0
|   |--- Cloud_Cover >  50.03
|   |   |--- Temperature <= 24.91
|   |   |   |--- class: 1
|   |   |--- Temperature >  24.91
|   |   |   |--- class: 0



In [17]:
#  Accuracy & Report
y_pred = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%') 
print(classification_report(y_test, y_pred, target_names=['no rain','rain'])) 

Accuracy: 100.0 %
              precision    recall  f1-score   support

     no rain       1.00      1.00      1.00       443
        rain       1.00      1.00      1.00        57

    accuracy                           1.00       500
   macro avg       1.00      1.00      1.00       500
weighted avg       1.00      1.00      1.00       500



In [ ]:
# Prediction on a New Sample
sample = pd.DataFrame([{'Temperature': 5, 'Humidity': 20, 'Wind_Speed': 4, 'Cloud_Cover': 25, 'Pressure': 1005}])
pred = model.predict(sample)
print('Prediction:', 'rain' if pred[0]==1 else 'no rain')

Prediction: no rain


In [18]:
# Prediction on another Sample
sample2 = pd.DataFrame([{
    'Temperature': 24,     
    'Humidity': 85,         
    'Wind_Speed': 6,        
    'Cloud_Cover': 60,     
    'Pressure': 1005     
}])

pred = model.predict(sample2)
print('Prediction:', 'rain' if pred[0]==1 else 'no rain')

Prediction: rain
